# Cohort 1 versus cohort 3

Side-by-side trial/block distributions, time courses, problem averages, pooled true-value rank proportions, and total reversals for problems 1–3. Columns are cohorts and panels share scales. Full-data results are separate from the additional problem-2 comparison capped at 510 trials (shorter records retain all trials and are flagged).

Time is days since each mouse’s first observed session in a problem; daily means combine multiple observations. Block lengths include observed edge portions. Cohort means give mice equal weight. Rank proportions use the existing block-quality filters. These are descriptive comparisons, not paired tests between cohorts.


In [ ]:
import importlib
import sys
from pathlib import Path
from unittest.mock import patch
repo_root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                  if (p / "src" / "behavior_import").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Open this notebook within the analysis repository.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
comparison_problems = (1, 2, 3)
comparison_trial_cap = 510  # Additional, separate problem-2 comparison; None disables it.
show_comparison_plots = True
raw_root = Path("/Volumes/behrens/meg/3x3_field_magnitude_bandit/rawdata")
cohort_labels = ("cohort-01", "cohort-03")
excluded_task_name_parts = ("mag", "pretraining")


In [ ]:
import_data_module = importlib.import_module("src.behavior_import.import_data")
extraction = importlib.reload(importlib.import_module("src.behavior_import.extract_trials"))
from src.behavior_import.extract_trials import group_sessions_by_problem
from src.behavior_analysis.trial_block_summary import subject_numbers_from_paths
cohort_problem_data = {}
cohort_subject_numbers = {}
for label in cohort_labels:
    root = raw_root / label
    if not root.is_dir():
        raise FileNotFoundError(f"Mount the data drive: {root}")
    eligible = []
    for path, df in import_data_module.collect_tsv_files(root):
        names = df.loc[(df["type"] == "info") & (df["subtype"] == "task_name"), "content"]
        if len(names) and not any(x in str(names.iloc[0]).casefold() for x in excluded_task_name_parts):
            eligible.append((path, df))
    cohort_subject_numbers[label] = subject_numbers_from_paths(path for path, _ in eligible)
    with patch.object(import_data_module, "collect_tsv_files", return_value=eligible):
        raw = import_data_module.import_data(root)
    extracted = extraction.extract_trials(raw)
    problems = group_sessions_by_problem(extracted, copy_sessions=False)
    missing = set(comparison_problems) - set(problems)
    if missing:
        raise ValueError(f"{label}: missing problems {sorted(missing)}")
    cohort_problem_data[label] = {p: problems[p] for p in comparison_problems}
    print(f"Loaded {label}: problems {sorted(cohort_problem_data[label])}")


In [ ]:
comparison_plotting = importlib.reload(importlib.import_module("src.behavior_visualization.plot_cohort_comparison"))
comparison_dir = repo_root / "results" / "figures" / "open-field" / "cohort-01-vs-cohort-03" / ("problems-" + "-".join(map(str, comparison_problems)))
comparison_paths = comparison_plotting.plot_cohort_comparison(
    cohort_problem_data, cohort_subject_numbers, comparison_dir,
    problems=comparison_problems, trial_budget=comparison_trial_cap,
    show=show_comparison_plots)
print(f"Saved {len(comparison_paths)//2} comparisons as PNG/PDF, plus source tables, to {comparison_dir}")
